In [1]:
from llama_cpp import Llama

# You must download the specific .gguf file from the Hugging Face repo locally first.
# e.g., MN-12B-Starcannon-v3.Q4_K_M.gguf
model = Llama(
    model_path = "./MN-12B-Starcannon-v3.Q4_K_M.gguf",
    n_ctx = 4096,      # Context window size
    n_gpu_layers = -1,  # Offload all layers to GPU
    verbose = False
)

output = model("What is the capital of France?", max_tokens=32)
print(output["choices"][0]["text"])

llama_context: n_ctx_seq (4096) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


'' and the like. (The answer is Paris.) The French are the most nationalist people in Europe, and their nationalism is more intense than that of the Germans


In [2]:
# Time counter decorator 

import time
from functools import wraps

def time_calculator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t1 = time.time()
        result = func(*args, **kwargs)
        t2 = time.time()
        diff = t2 - t1
        print(f"🕑 [{func.__name__}] executed in {diff:.4f} seconds")
        return result
    return wrapper


In [3]:
import textwrap

@time_calculator
def chat(usr_message, max_tokens=1024):
    output = model.create_chat_completion(
        messages=[
            {
                "role": "system", 
                "content": (
                    "You are a strict data extraction API. You only output valid JSON. "
                    "Never output conversational text, markdown formatting, or explanations. "
                    "If a field cannot be found, use null. "
                    "Your output must exactly match this JSON schema: "
                    "{'name': string, 'age': number, 'profession': string, 'hobbies': list[string]}"
                )
            },
            {"role": "user", "content": usr_message}
        ],
        max_tokens=max_tokens
    )

    
    response = output["choices"][0]["message"]["content"]
    input_tokens = output["usage"]["prompt_tokens"]
    output_tokens = output["usage"]["completion_tokens"]

    # Wrap text to 100 characters while preserving paragraph breaks
    for paragraph in response.split('\n'):
        print(textwrap.fill(paragraph, width=100))
        
    print(f"\n[Tokens] Input: {input_tokens} | Output: {output_tokens}")
    
    return response, input_tokens, output_tokens


## Test 1: Standard Extraction

"John is a 34-year-old software engineer who spends his weekends hiking and reading sci-fi novels." Goal: The model should output perfect JSON without missing any keys.



In [4]:
chat("John is a 34-year-old software engineer who spends his weekends hiking and reading sci-fi novels.")

{
  "name": "John",
  "age": 34,
  "profession": "software engineer",
  "hobbies": [
    "hiking",
    "reading sci-fi novels"
  ]
}

[Tokens] Input: 104 | Output: 47
🕑 [chat] executed in 1.7078 seconds


('{\n  "name": "John",\n  "age": 34,\n  "profession": "software engineer",\n  "hobbies": [\n    "hiking",\n    "reading sci-fi novels"\n  ]\n}',
 104,
 47)

## Test 2: Missing Data (Hallucination Check)
"Sarah loves painting and playing the guitar." 

Goal: The model should output null for the age and profession keys. Less capable models will hallucinate an age or drop the keys entirely.



In [5]:
chat("Sarah loves painting and playing the guitar.")

{
  "name": "Sarah",
  "age": null,
  "profession": null,
  "hobbies": [
    "painting",
    "playing the guitar"
  ]
}

[Tokens] Input: 91 | Output: 43
🕑 [chat] executed in 1.3978 seconds


('{\n  "name": "Sarah",\n  "age": null,\n  "profession": null,\n  "hobbies": [\n    "painting",\n    "playing the guitar"\n  ]\n}',
 91,
 43)

## Test 3: The Conversational Trap (Strictness Check)

"Hey! I really need you to extract this into JSON for me, it's super important. The guy's name is Michael, he's 40, a doctor, and he likes golf. Can you format that?" 

Goal: The model should output ONLY the JSON. Weak models will reply with "Sure, here is the JSON you requested:" before the curly braces, which breaks automated parsing.



In [6]:
chat("Hey! I really need you to extract this into JSON for me, it's super important. The guy's name is Michael, he's 40, a doctor, and he likes golf. Can you format that?")

{
  "name": "Michael",
  "age": 40,
  "profession": "doctor",
  "hobbies": [
    "golf"
  ]
}

[Tokens] Input: 128 | Output: 39
🕑 [chat] executed in 1.3487 seconds


('{\n  "name": "Michael",\n  "age": 40,\n  "profession": "doctor",\n  "hobbies": [\n    "golf"\n  ]\n}',
 128,
 39)